# RNN, LSTM, GRU 활용 예제: Image Captioning, TTS, Translation

이번 주 강의에서는 RNN이 시계열/순차 데이터를 다루는 방식과, 그 한계를 해결한
**LSTM**, **GRU**를 배웠습니다. 이 노트북에서는 LSTM이 실제로 어떻게 쓰이는지
세 가지 예제로 직접 확인합니다.

| Part | 예제 | 핵심 아이디어 |
|---|---|---|
| **Part 1. Image Captioning** | 사진 → 문장 | CNN이 뽑은 이미지 특징을 조건으로, LSTM이 단어를 한 개씩 생성 |
| **Part 2. Text-to-Speech (TTS)** | 문장 → 음성 | LSTM 기반 디코더(Tacotron2)가 텍스트로부터 음성 스펙트로그램을 생성 |
| **Part 3. Translation** | 문장 → 번역된 문장 | GRU Encoder-Decoder + Attention (Seq2Seq)으로 번역 수행 |

**학습 목표**
- CNN(인코더) + LSTM(디코더) 구조로 이미지로부터 문장을 생성하는 과정을 이해한다.
- 사전학습된 LSTM 기반 TTS 모델로 텍스트를 음성으로 합성하는 과정을 경험한다.
- GRU 기반 Seq2Seq + Attention을 직접 학습시켜보고, 실제 번역 서비스와 비교해본다.

**실행 환경:** Google Colab (`런타임 > 런타임 유형 변경`에서 GPU 선택을 권장합니다).
Part 1, 3은 노트북 안에서 직접 학습을 진행하므로, 전체 노트북을 처음부터 끝까지
실행하는 데 GPU 기준 약 15~20분 정도 걸립니다. Part 1, 2, 3은 Part 0만
실행되어 있으면 서로 독립적으로, 어떤 순서로든 실행할 수 있습니다.

## Part 0. 환경 설정

아래 셀은 모든 Part에서 공통으로 쓰는 라이브러리 설치/임포트와 random seed
고정입니다. **반드시 가장 먼저 한 번 실행**하세요.

In [ ]:
# torchaudio(Part 2)와 deep-translator(Part 3)는 Colab에 기본 설치되어 있지 않을 수 있으므로 설치합니다.
# fonts-nanum은 Part 3의 그래프에서 한글이 깨지지 않고 보이도록 설치하는 한글 폰트입니다.
!pip install -q torchaudio deep-translator
!apt-get -qq update && apt-get -qq install -y fonts-nanum > /dev/null 2>&1

In [ ]:
import os
import re
import json
import time
import random
import urllib.request
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image

print("PyTorch version:", torch.__version__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Part 3에서 그래프에 한글(번역 결과)을 표시하므로, matplotlib 기본 폰트에는
# 한글 글리프가 없어 그대로 두면 네모(tofu)로 깨져 보입니다. 위에서 설치한
# 나눔고딕 폰트를 등록해서 한글이 정상적으로 보이도록 설정합니다.
try:
    for font_path in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
        fm.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 폰트(NanumGothic) 설정 완료.")
except Exception as e:
    print(f"한글 폰트 설정에 실패했습니다 ({e}). Part 3 그래프의 한글이 깨져 보일 수 있습니다.")

## Part 1. Image Captioning: CNN + LSTM으로 사진을 문장으로 설명하기

### 목표

슬라이드에서 본 "남자가 / 파도 위에서 / 서핑을" 예제처럼, 이미지 하나를 보고
그 내용을 설명하는 문장을 **한 단어씩 순차적으로 생성**하는 모델을 직접
학습시켜 봅니다. 이것은 전형적인 **One-to-Many RNN 구조**입니다: 입력은
"이미지 하나"(고정 길이)지만, 출력은 "단어들의 시퀀스"(가변 길이)입니다.

**구조:** 사전학습된 CNN(ResNet18)이 이미지에서 특징 벡터를 뽑고, 이 벡터를
**LSTM의 초기 상태(h0, c0)** 로 사용해 문장을 한 단어씩 생성합니다. CNN은
Week 9에서 썼던 것과 동일하게 ImageNet으로 사전학습된 것을 그대로(freeze)
사용하고, **LSTM 디코더만 이번 노트북에서 직접 학습**시킵니다.

### 데이터 준비

실제 [COCO](https://cocodataset.org) 데이터셋의 캡션(이미지 설명) 중 일부를
사용합니다. 전체 COCO는 수십만 장이라 학습이 오래 걸리므로, 실행 시간을
줄이기 위해 이미지 수(`N_IMAGES`)를 제한합니다. 먼저 캡션이 들어 있는
annotation 파일을 내려받고, 그중 사용할 이미지만 골라 다운로드합니다.

In [ ]:
# 실행 시간 조절용 하이퍼파라미터
N_IMAGES = 2000       # 학습에 사용할 이미지 수 (전체 COCO val2017은 5,000장)
NUM_EPOCHS = 12
BATCH_SIZE = 64
EMBED_DIM = 256
HIDDEN_DIM = 512
MIN_WORD_FREQ = 4     # 이보다 적게 등장한 단어는 어휘집에서 제외 (<unk> 처리)

COCO_DIR = "coco_captioning"
IMG_DIR = os.path.join(COCO_DIR, "images")
os.makedirs(IMG_DIR, exist_ok=True)

ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ann_zip_path = os.path.join(COCO_DIR, "annotations_trainval2017.zip")
if not os.path.exists(ann_zip_path):
    print("캡션 annotation 파일을 내려받는 중... (약 240MB, 몇 분 정도 걸릴 수 있습니다)")
    urllib.request.urlretrieve(ANNOTATIONS_URL, ann_zip_path)

import zipfile
with zipfile.ZipFile(ann_zip_path) as zf:
    with zf.open("annotations/captions_val2017.json") as f:
        captions_data = json.load(f)

os.remove(ann_zip_path)  # 캡션만 쓰고 나머지(instances, keypoints 등)는 필요 없으므로 지웁니다.
print("전체 이미지 수:", len(captions_data["images"]))
print("전체 캡션 수:", len(captions_data["annotations"]))

In [ ]:
id_to_filename = {img["id"]: img["file_name"] for img in captions_data["images"]}
id_to_url = {img["id"]: img["coco_url"] for img in captions_data["images"]}

id_to_captions = defaultdict(list)
for ann in captions_data["annotations"]:
    id_to_captions[ann["image_id"]].append(ann["caption"])

all_ids = sorted(id_to_filename.keys())
rng = random.Random(SEED)
rng.shuffle(all_ids)
subset_ids = all_ids[:N_IMAGES]
print(f"이번 학습에 사용할 이미지: {len(subset_ids)}장")

In [ ]:
import concurrent.futures as cf

def download_one(image_id):
    fname = id_to_filename[image_id]
    dst = os.path.join(IMG_DIR, fname)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(id_to_url[image_id], dst)
    return image_id

print("이미지를 내려받는 중... (수천 개의 작은 파일이라 시간이 걸릴 수 있습니다)")
start_time = time.time()
with cf.ThreadPoolExecutor(max_workers=16) as executor:
    list(executor.map(download_one, subset_ids))
print(f"다운로드 완료: {time.time() - start_time:.1f}초")

### 어휘집(Vocabulary) 만들기

LSTM은 단어를 직접 이해하지 못하므로, 모든 단어를 정수 인덱스로 바꿔야
합니다. 소문자로 통일하고 알파벳만 남긴 뒤, 너무 드물게 등장하는 단어는
`<unk>`(unknown)로 처리해 어휘집 크기를 적당히 유지합니다.

In [ ]:
def tokenize(caption):
    return re.findall(r"[a-z]+", caption.lower())


word_freq = Counter()
for image_id in subset_ids:
    for caption in id_to_captions[image_id]:
        word_freq.update(tokenize(caption))

specials = ["<pad>", "<start>", "<end>", "<unk>"]
vocab_words = specials + [w for w, c in word_freq.items() if c >= MIN_WORD_FREQ]
word_to_idx = {w: i for i, w in enumerate(vocab_words)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
PAD, START, END, UNK = (word_to_idx[t] for t in specials)

print(f"어휘집 크기: {len(vocab_words)}개 단어 (최소 {MIN_WORD_FREQ}회 이상 등장)")


def encode_caption(caption, max_len=20):
    tokens = tokenize(caption)[: max_len - 2]
    ids = [START] + [word_to_idx.get(t, UNK) for t in tokens] + [END]
    ids += [PAD] * (max_len - len(ids))
    return ids

### CNN으로 이미지 특징 미리 뽑아두기

이미지마다 캡션이 여러 개 있어서, 같은 이미지를 반복해서 CNN에 통과시키면
낭비입니다. 그래서 이미지당 **딱 한 번만** CNN을 통과시켜 특징 벡터를
미리 계산해두고(caching), 이후 LSTM 학습에서는 이 벡터만 재사용합니다 —
이렇게 하면 한 epoch이 훨씬 빨라집니다.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import transforms as T

cnn_weights = ResNet18_Weights.IMAGENET1K_V1
encoder_cnn = resnet18(weights=cnn_weights)
encoder_cnn.fc = nn.Identity()  # 마지막 분류 층을 제거하면 512차원 특징 벡터가 바로 나옵니다.
encoder_cnn = encoder_cnn.to(device).eval()
for param in encoder_cnn.parameters():
    param.requires_grad = False  # CNN은 그대로 사용(freeze)하고 학습하지 않습니다.

FEATURE_DIM = 512
preprocess = cnn_weights.transforms()


@torch.no_grad()
def extract_feature(image_path):
    image = Image.open(image_path).convert("RGB")
    tensor = preprocess(image).unsqueeze(0).to(device)
    return encoder_cnn(tensor).squeeze(0).cpu()


print("이미지 특징을 미리 계산하는 중...")
start_time = time.time()
features_by_id = {}
for image_id in subset_ids:
    path = os.path.join(IMG_DIR, id_to_filename[image_id])
    features_by_id[image_id] = extract_feature(path)
print(f"완료: {time.time() - start_time:.1f}초, 이미지 {len(features_by_id)}장")

### 학습 데이터셋 구성 및 Train/Validation 분리

이미지 하나당 캡션이 여러 개이므로, (이미지 특징, 캡션 하나) 쌍을 모두
풀어서 학습 샘플로 사용합니다.

In [ ]:
MAX_CAPTION_LEN = 20

pairs = []  # (image_id, encoded_caption)
for image_id in subset_ids:
    for caption in id_to_captions[image_id]:
        pairs.append((image_id, encode_caption(caption, max_len=MAX_CAPTION_LEN)))

rng.shuffle(pairs)
n_val = max(1, int(len(pairs) * 0.1))
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
print(f"학습 샘플: {len(train_pairs)}개, 검증 샘플: {len(val_pairs)}개")


class CaptionDataset(torch.utils.data.Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_id, caption_ids = self.pairs[idx]
        return features_by_id[image_id], torch.tensor(caption_ids, dtype=torch.long)


train_loader = torch.utils.data.DataLoader(CaptionDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(CaptionDataset(val_pairs), batch_size=BATCH_SIZE, shuffle=False)

### LSTM 디코더 정의

CNN 특징 벡터를 LSTM의 초기 hidden/cell state로 바꾸는 작은 선형 층을 두고,
그 상태에서 시작해 `<start>` 토큰부터 한 단어씩 다음 단어를 예측하도록
학습합니다 (Teacher Forcing: 학습 중에는 모델이 직접 생성한 단어 대신
정답 캡션을 다음 입력으로 사용합니다).

In [ ]:
class CaptionDecoder(nn.Module):
    def __init__(self, feature_dim, embed_dim, hidden_dim, vocab_size):
        super().__init__()
        self.init_h = nn.Linear(feature_dim, hidden_dim)
        self.init_c = nn.Linear(feature_dim, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, input_tokens):
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)
        embedded = self.embedding(input_tokens)
        outputs, _ = self.lstm(embedded, (h0, c0))
        return self.fc(outputs)

    @torch.no_grad()
    def generate(self, feature, max_len=20):
        self.eval()
        h = self.init_h(feature.unsqueeze(0)).unsqueeze(0)
        c = self.init_c(feature.unsqueeze(0)).unsqueeze(0)
        input_token = torch.tensor([[START]], device=feature.device)
        words = []
        for _ in range(max_len):
            embedded = self.embedding(input_token)
            output, (h, c) = self.lstm(embedded, (h, c))
            next_token = self.fc(output[:, -1]).argmax(dim=-1).item()
            if next_token == END:
                break
            words.append(idx_to_word[next_token])
            input_token = torch.tensor([[next_token]], device=feature.device)
        return " ".join(words)


caption_decoder = CaptionDecoder(FEATURE_DIM, EMBED_DIM, HIDDEN_DIM, len(vocab_words)).to(device)
caption_criterion = nn.CrossEntropyLoss(ignore_index=PAD)
caption_optimizer = torch.optim.Adam(caption_decoder.parameters(), lr=1e-3)

### 학습

In [ ]:
def run_caption_epoch(loader, train: bool):
    caption_decoder.train(train)
    total_loss, total_tokens = 0.0, 0
    for features, captions in loader:
        features, captions = features.to(device), captions.to(device)
        input_tokens, targets = captions[:, :-1], captions[:, 1:]

        logits = caption_decoder(features, input_tokens)
        loss = caption_criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        if train:
            caption_optimizer.zero_grad()
            loss.backward()
            caption_optimizer.step()

        n_tokens = (targets != PAD).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens
    return total_loss / total_tokens


caption_history = {"train_loss": [], "val_loss": []}
for epoch in range(NUM_EPOCHS):
    train_loss = run_caption_epoch(train_loader, train=True)
    val_loss = run_caption_epoch(val_loader, train=False)
    caption_history["train_loss"].append(train_loss)
    caption_history["val_loss"].append(val_loss)
    print(f"epoch {epoch + 1}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(caption_history["train_loss"], label="train")
plt.plot(caption_history["val_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Image Captioning LSTM Decoder Training")
plt.legend()
plt.show()

### 직접 캡션 생성해보기

기본으로는 Week 9에서 썼던 `zidane.jpg` 이미지를 사용합니다. Colab에서
실행 중이라면, 원하는 이미지를 직접 업로드해서 캡션을 생성해볼 수도
있습니다.

In [ ]:
DEFAULT_IMAGE_URL = "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/zidane.jpg"
demo_image_path = "caption_demo.jpg"
urllib.request.urlretrieve(DEFAULT_IMAGE_URL, demo_image_path)

# Colab에서는 직접 업로드한 이미지로 바꿔서 캡션을 생성해볼 수 있습니다.
try:
    from google.colab import files
    print("캡션을 생성할 이미지를 업로드하세요 (취소하려면 Cancel upload를 누르세요).")
    uploaded = files.upload()
    if uploaded:
        demo_image_path = list(uploaded.keys())[0]
except ImportError:
    print("Google Colab 환경이 아니므로 업로드를 건너뜁니다. 기본 이미지로 계속합니다.")

demo_feature = extract_feature(demo_image_path).to(device)
generated_caption = caption_decoder.generate(demo_feature, max_len=MAX_CAPTION_LEN)

plt.figure(figsize=(6, 6))
plt.imshow(Image.open(demo_image_path).convert("RGB"))
plt.title(f"Generated caption: {generated_caption}")
plt.axis("off")
plt.show()

### 참고할 점

이 모델은 수업 시간에 짧게(2,000장, 12 epoch) 학습시킨 작은 LSTM입니다.
실제 서비스에 쓰이는 이미지 캡셔닝 모델은 COCO 전체(약 12만 장)를 수십
epoch 학습시키거나, 요즘은 Transformer 기반 구조(BLIP, GIT 등)를 사용합니다.
그래도 CNN이 뽑은 이미지 정보를 바탕으로 LSTM이 그럴듯한 단어 순서를
만들어낸다는 핵심 아이디어는 동일합니다.

In [ ]:
# 다음 Part로 넘어가기 전에 메모리를 정리합니다.
del encoder_cnn, caption_decoder
if device.type == "cuda":
    torch.cuda.empty_cache()

## Part 2. Text-to-Speech: LSTM 디코더로 문장을 음성으로 합성하기

### 목표

슬라이드에서 "LSTM이 Transformer 이전에 널리 쓰인 분야"로 소개된 **Speech
Synthesis(TTS)** 를 직접 체험해봅니다. 문장을 입력하면, 사전학습된
**Tacotron2** 모델이 그 문장을 소리로 바꿔줍니다.

### 개념 설명

- Tacotron2는 텍스트를 받아 **멜 스펙트로그램**(음성의 주파수 정보를 담은
  2차원 표현)을 생성하는 모델입니다. 이 생성 과정의 핵심이 바로 **LSTM
  디코더**입니다 — 이전에 생성한 프레임 정보를 바탕으로 다음 프레임을 한
  단계씩 순차적으로 만들어낸다는 점에서, Part 1의 캡션 생성과 구조가
  비슷합니다 (다만 출력이 단어가 아니라 음성 프레임입니다).
- 멜 스펙트로그램만으로는 스피커에서 소리가 나지 않으므로, **보코더(vocoder)**
  가 이를 실제 파형(waveform)으로 변환합니다. 여기서는 별도 학습이 필요 없는
  **Griffin-Lim** 알고리즘을 보코더로 사용합니다 (신호 처리 기법으로, 속도는
  빠르지만 음질은 신경망 기반 보코더보다 다소 떨어집니다).
- 이 모델은 영어 음성 데이터(LJSpeech)로 학습되었으므로, **영어 문장만**
  자연스럽게 합성됩니다.

In [ ]:
import torchaudio
from torchaudio.pipelines import TACOTRON2_GRIFFINLIM_CHAR_LJSPEECH

tts_bundle = TACOTRON2_GRIFFINLIM_CHAR_LJSPEECH
tts_processor = tts_bundle.get_text_processor()
tts_model = tts_bundle.get_tacotron2().to(device).eval()
tts_vocoder = tts_bundle.get_vocoder().to(device).eval()
print("Tacotron2 로딩 완료. Sample rate:", tts_vocoder.sample_rate)

### 문장을 입력하고 들어보기

아래 `sentence`를 원하는 영어 문장으로 바꿔서 실행해보세요.

In [ ]:
sentence = "Long short term memory networks are a special kind of recurrent neural network."

with torch.inference_mode():
    processed_text, text_lengths = tts_processor(sentence)
    processed_text, text_lengths = processed_text.to(device), text_lengths.to(device)
    mel_spectrogram, mel_lengths, _ = tts_model.infer(processed_text, text_lengths)
    waveform, waveform_lengths = tts_vocoder(mel_spectrogram, mel_lengths)

print(f"생성된 오디오 길이: {waveform.shape[-1] / tts_vocoder.sample_rate:.2f}초")

fig, axes = plt.subplots(2, 1, figsize=(9, 5))
axes[0].imshow(mel_spectrogram[0].cpu().numpy(), origin="lower", aspect="auto")
axes[0].set_title("Mel Spectrogram (LSTM decoder output)")
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Mel channel")
axes[1].plot(waveform[0].cpu().numpy())
axes[1].set_title("Waveform (Griffin-Lim vocoder output)")
axes[1].set_xlabel("Sample")
plt.tight_layout()
plt.show()

from IPython.display import Audio, display
display(Audio(waveform[0].cpu().numpy(), rate=tts_vocoder.sample_rate))

### 참고할 점

여기서 쓴 Griffin-Lim 보코더는 추가 학습 없이 빠르게 파형을 만들어내지만,
사람 목소리처럼 자연스럽진 않고 다소 기계적으로 들립니다. 실제 서비스
(예: Google TTS, 애플 Siri 등)는 WaveNet, HiFi-GAN 같은 **신경망 기반
보코더**를 사용해 훨씬 자연스러운 음성을 만들어냅니다.

In [ ]:
# 다음 Part로 넘어가기 전에 메모리를 정리합니다.
del tts_model, tts_vocoder
if device.type == "cuda":
    torch.cuda.empty_cache()

## Part 3. Translation: GRU Seq2Seq + Attention을 직접 학습시켜보기

### 목표

슬라이드에서 다룬 영어→한국어 번역 예제("I love you" → "나는 너를 사랑해")를
그대로 구현합니다. **GRU 기반 Encoder-Decoder(Seq2Seq)** 구조에 **Attention**
을 추가해 영어 문장을 한국어로 번역하는 모델을 직접 학습시키고, 실제
**Google Translate**의 번역 결과와 비교해봅니다.

### 개념 설명

- **Encoder**: 영어 문장을 단어 하나씩 GRU에 통과시켜, 각 시점의 hidden
  state를 모두 기록해둡니다 (Many-to-Many 구조).
- **Decoder**: 한국어 단어를 한 개씩 생성합니다 (One-to-Many 구조). 이때
  매 단어를 생성할 때마다 **Attention**이 encoder의 모든 hidden state 중
  지금 가장 관련 있는 부분에 가중치를 더 많이 줘서 참고합니다 — 슬라이드의
  "h4를 모든 ⊙에 전달" 그림과 동일한 방식입니다.
- 이 모델은 **수업 시간에 짧게** 학습시킨 작은 모델이라, 학습 데이터와
  비슷한 짧고 단순한 문장은 꽤 그럴듯하게 번역하지만, 학습 데이터에 없던
  단어가 섞인 문장은 번역이 많이 틀릴 수 있습니다. 바로 이 지점에서 실제
  Google Translate(수십억 개의 문장으로 학습된 production 시스템)와의 격차를
  직접 눈으로 확인하는 것이 이 Part의 핵심입니다.

### 데이터 준비

실제 영어-한국어 병렬 문장 데이터셋인 [OPUS-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100)
(en-ko)에서, 학습 속도를 위해 **짧은 문장들만** 골라 일부만 사용합니다.

In [ ]:
# 실행 시간 조절용 하이퍼파라미터
MAX_WORDS = 10         # 문장당 최대 단어 수 (길면 제외)
SUBSET_SIZE = 8000     # 사용할 문장 쌍 개수
MT_NUM_EPOCHS = 10
MT_BATCH_SIZE = 64
MT_EMBED_DIM = 256
MT_HIDDEN_DIM = 512
MIN_TOKEN_FREQ = 2

from datasets import load_dataset

# streaming=True: 전체 데이터(수십만 문장)를 다 내려받지 않고,
# 조건에 맞는 문장을 필요한 만큼만 받아옵니다.
raw_dataset = load_dataset("Helsinki-NLP/opus-100", "en-ko", split="train", streaming=True)


def en_tokenize(text):
    return re.findall(r"[a-zA-Z']+", text.lower())


def ko_tokenize(text):
    return text.strip().split()


filtered_pairs = []
for example in raw_dataset:
    en_tokens = en_tokenize(example["translation"]["en"])
    ko_tokens = ko_tokenize(example["translation"]["ko"])
    if 1 <= len(en_tokens) <= MAX_WORDS and 1 <= len(ko_tokens) <= MAX_WORDS:
        filtered_pairs.append((en_tokens, ko_tokens))
    if len(filtered_pairs) >= SUBSET_SIZE:
        break

print(f"필터링 후 사용할 문장 쌍: {len(filtered_pairs)}개")
print("예시:", " ".join(filtered_pairs[0][0]), "->", " ".join(filtered_pairs[0][1]))

### 어휘집 만들기 (영어, 한국어 각각)

이번에는 소스(영어)와 타겟(한국어) 언어가 다르므로, 어휘집을 각각 따로
만듭니다. 한국어는 형태소 분석 없이 **공백 기준으로만** 나눕니다 — 완벽한
방식은 아니지만, 토이 모델 수준에서는 충분합니다.

In [ ]:
def build_vocab(tokenized_sentences, min_freq):
    freq = Counter()
    for tokens in tokenized_sentences:
        freq.update(tokens)
    specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
    words = specials + [w for w, c in freq.items() if c >= min_freq]
    return {w: i for i, w in enumerate(words)}


en_vocab = build_vocab([p[0] for p in filtered_pairs], MIN_TOKEN_FREQ)
ko_vocab = build_vocab([p[1] for p in filtered_pairs], MIN_TOKEN_FREQ)
ko_idx_to_word = {i: w for w, i in ko_vocab.items()}

EN_PAD, EN_SOS, EN_EOS, EN_UNK = (en_vocab[t] for t in ["<pad>", "<sos>", "<eos>", "<unk>"])
KO_PAD, KO_SOS, KO_EOS, KO_UNK = (ko_vocab[t] for t in ["<pad>", "<sos>", "<eos>", "<unk>"])

print(f"영어 어휘집 크기: {len(en_vocab)}, 한국어 어휘집 크기: {len(ko_vocab)}")

MT_MAX_LEN = MAX_WORDS + 2  # <sos>, <eos> 포함


def encode(tokens, vocab, unk_idx, sos_idx, eos_idx):
    ids = [sos_idx] + [vocab.get(t, unk_idx) for t in tokens] + [eos_idx]
    ids += [vocab["<pad>"]] * (MT_MAX_LEN - len(ids))
    return ids[:MT_MAX_LEN]

### Dataset 구성 및 Train/Validation 분리

In [ ]:
encoded_pairs = [
    (encode(en, en_vocab, EN_UNK, EN_SOS, EN_EOS), encode(ko, ko_vocab, KO_UNK, KO_SOS, KO_EOS))
    for en, ko in filtered_pairs
]

mt_rng = random.Random(SEED)
mt_rng.shuffle(encoded_pairs)
n_val = max(1, int(len(encoded_pairs) * 0.1))
mt_val_pairs, mt_train_pairs = encoded_pairs[:n_val], encoded_pairs[n_val:]
print(f"학습 샘플: {len(mt_train_pairs)}개, 검증 샘플: {len(mt_val_pairs)}개")


class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)


mt_train_loader = torch.utils.data.DataLoader(TranslationDataset(mt_train_pairs), batch_size=MT_BATCH_SIZE, shuffle=True)
mt_val_loader = torch.utils.data.DataLoader(TranslationDataset(mt_val_pairs), batch_size=MT_BATCH_SIZE, shuffle=False)

### GRU Encoder, Attention, GRU Decoder 정의

In [ ]:
class EncoderGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=EN_PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, input_seq):
        embedded = self.embedding(input_seq)
        outputs, hidden = self.gru(embedded)  # outputs: (B, T, H), hidden: (1, B, H)
        return outputs, hidden


class Attention(nn.Module):
    """Bahdanau-style additive attention: decoder의 현재 상태와 encoder의 모든 시점을 비교해 가중치를 만듭니다."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, pad_mask):
        T = encoder_outputs.size(1)
        hidden_expanded = decoder_hidden.permute(1, 0, 2).repeat(1, T, 1)  # (B, T, H)
        energy = torch.tanh(self.attn(torch.cat([hidden_expanded, encoder_outputs], dim=2)))
        scores = self.v(energy).squeeze(2)  # (B, T)
        scores = scores.masked_fill(pad_mask == 0, float("-inf"))
        return torch.softmax(scores, dim=1)  # (B, T), 합이 1인 attention weight


class DecoderGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=KO_PAD)
        self.attention = Attention(hidden_dim)
        self.gru = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2 + embed_dim, vocab_size)

    def forward(self, input_token, hidden, encoder_outputs, pad_mask):
        embedded = self.embedding(input_token)  # (B, 1, E)
        attn_weights = self.attention(hidden, encoder_outputs, pad_mask)  # (B, T)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)  # (B, 1, H)
        gru_input = torch.cat([embedded, context], dim=2)
        output, hidden = self.gru(gru_input, hidden)
        combined = torch.cat([output.squeeze(1), context.squeeze(1), embedded.squeeze(1)], dim=1)
        logits = self.fc(combined)
        return logits, hidden, attn_weights


mt_encoder = EncoderGRU(len(en_vocab), MT_EMBED_DIM, MT_HIDDEN_DIM).to(device)
mt_decoder = DecoderGRU(len(ko_vocab), MT_EMBED_DIM, MT_HIDDEN_DIM).to(device)
mt_criterion = nn.CrossEntropyLoss(ignore_index=KO_PAD)
mt_optimizer = torch.optim.Adam(list(mt_encoder.parameters()) + list(mt_decoder.parameters()), lr=1e-3)

### 학습 (Teacher Forcing)

In [ ]:
def run_translation_batch(src, tgt, train: bool):
    src, tgt = src.to(device), tgt.to(device)
    pad_mask = (src != EN_PAD).float()

    encoder_outputs, hidden = mt_encoder(src)

    decoder_input = tgt[:, :1]  # <sos>
    all_logits = []
    for t in range(1, tgt.size(1)):
        logits, hidden, _ = mt_decoder(decoder_input, hidden, encoder_outputs, pad_mask)
        all_logits.append(logits)
        decoder_input = tgt[:, t : t + 1]  # teacher forcing: 정답을 다음 입력으로 사용

    # 시점별로 loss를 따로 구해 더하면, 짧은 문장들만 모인 배치에서는 뒤쪽
    # 시점의 target이 전부 <pad>가 되어 0/0(NaN)이 될 수 있습니다. 모든
    # 시점의 logits/target을 한 번에 펼쳐서 CrossEntropyLoss를 단 한 번만
    # 호출하면, batch 전체에 유효한 토큰이 하나라도 있는 한 안전합니다.
    all_logits = torch.stack(all_logits, dim=1)  # (B, T-1, V)
    targets = tgt[:, 1:]  # (B, T-1)
    loss = mt_criterion(all_logits.reshape(-1, all_logits.size(-1)), targets.reshape(-1))

    if train:
        mt_optimizer.zero_grad()
        loss.backward()
        mt_optimizer.step()

    return loss.item()


def run_translation_epoch(loader, train: bool):
    mt_encoder.train(train)
    mt_decoder.train(train)
    total_loss, total_batches = 0.0, 0
    for src, tgt in loader:
        total_loss += run_translation_batch(src, tgt, train)
        total_batches += 1
    return total_loss / total_batches


mt_history = {"train_loss": [], "val_loss": []}
for epoch in range(MT_NUM_EPOCHS):
    train_loss = run_translation_epoch(mt_train_loader, train=True)
    val_loss = run_translation_epoch(mt_val_loader, train=False)
    mt_history["train_loss"].append(train_loss)
    mt_history["val_loss"].append(val_loss)
    print(f"epoch {epoch + 1}/{MT_NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(mt_history["train_loss"], label="train")
plt.plot(mt_history["val_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Sum Cross-Entropy Loss per Sentence")
plt.title("GRU Seq2Seq + Attention Training")
plt.legend()
plt.show()

### 직접 입력한 문장 번역해보기, Google Translate와 비교

아래 `sentence`를 원하는 영어 문장으로 바꿔서 실행해보세요. 학습 데이터에
나온 문장과 비슷하게 짧고 단순할수록 더 잘 번역될 가능성이 높습니다.

In [ ]:
@torch.no_grad()
def translate_sentence(sentence, max_len=MT_MAX_LEN):
    mt_encoder.eval()
    mt_decoder.eval()
    tokens = en_tokenize(sentence)
    src = torch.tensor([encode(tokens, en_vocab, EN_UNK, EN_SOS, EN_EOS)], device=device)
    pad_mask = (src != EN_PAD).float()

    encoder_outputs, hidden = mt_encoder(src)
    decoder_input = torch.tensor([[KO_SOS]], device=device)

    words = []
    attn_history = []
    for _ in range(max_len):
        logits, hidden, attn_weights = mt_decoder(decoder_input, hidden, encoder_outputs, pad_mask)
        next_token = logits.argmax(dim=-1).item()
        attn_history.append(attn_weights.squeeze(0).cpu().numpy())
        if next_token == KO_EOS:
            break
        words.append(ko_idx_to_word.get(next_token, "<unk>"))
        decoder_input = torch.tensor([[next_token]], device=device)
    return "".join(words), np.array(attn_history)  # 한국어는 띄어쓰기 없이 이어 붙입니다.


sentence = "I love you"

model_translation, attn_matrix = translate_sentence(sentence)
print(f"입력 문장:          {sentence}")
print(f"모델(GRU+Attention) 번역: {model_translation}")

try:
    from deep_translator import GoogleTranslator
    google_translation = GoogleTranslator(source="en", target="ko").translate(sentence)
    print(f"Google Translate 번역:   {google_translation}")
except Exception as e:
    print(f"Google Translate 호출에 실패했습니다 ({type(e).__name__}: {e}).")
    print("일시적인 요청 제한이거나 네트워크 문제일 수 있습니다. 잠시 후 다시 시도해보세요.")

### Attention 가중치 시각화

디코더가 각 한국어 단어를 생성할 때 영어 문장의 어느 부분에 더 주목했는지
확인해볼 수 있습니다 (밝을수록 더 많이 참고한 부분).

In [ ]:
if len(model_translation) > 0 and attn_matrix.size > 0:
    en_tokens_display = ["<sos>"] + en_tokenize(sentence) + ["<eos>"] + ["<pad>"] * MT_MAX_LEN
    en_tokens_display = en_tokens_display[: attn_matrix.shape[1]]
    ko_tokens_display = list(model_translation)

    plt.figure(figsize=(max(4, len(en_tokens_display) * 0.5), max(3, len(ko_tokens_display) * 0.5)))
    plt.imshow(attn_matrix[: len(ko_tokens_display)], aspect="auto", cmap="viridis")
    plt.xticks(range(len(en_tokens_display)), en_tokens_display, rotation=90)
    plt.yticks(range(len(ko_tokens_display)), ko_tokens_display)
    plt.xlabel("Input (English)")
    plt.ylabel("Output (Korean)")
    plt.title("Attention Weights")
    plt.colorbar()
    plt.tight_layout()
    plt.show()

### 참고할 점

이 모델은 8,000개의 짧은 문장만으로 짧게 학습한 토이(toy) 모델입니다.
Google Translate는 수십억 개의 문장과 훨씬 복잡한(대부분 Transformer 기반)
구조로 학습되어 있습니다. 두 결과의 차이가 클수록, "왜 최신 번역 시스템이
이만큼 많은 데이터와 연산을 필요로 하는지"를 보여주는 셈입니다.

In [ ]:
# 메모리 정리
del mt_encoder, mt_decoder
if device.type == "cuda":
    torch.cuda.empty_cache()